In [1]:
import json
import random
from pathlib import Path

random.seed(42)

CWD = Path.cwd()

if (CWD / "data").exists():
    BASE_DIR = CWD
else:
    BASE_DIR = CWD.parent

DATA_PROCESSED = BASE_DIR / "data" / "processed"

arquivo = DATA_PROCESSED / "fine_tuning_medico.jsonl"

dataset = []

with open(arquivo, "r", encoding="utf-8") as f:
    for linha in f:
        dataset.append(json.loads(linha))

print("Total carregado:", len(dataset))

Total carregado: 60


In [2]:
train_data = []
validation_data = []

categorias_unicas = sorted(set(item["categoria"] for item in dataset))

for categoria in categorias_unicas:

    itens_categoria = [
        item for item in dataset
        if item["categoria"] == categoria
    ]

    perguntas = list(set(
        item["messages"][1]["content"]
        for item in itens_categoria
    ))

    random.shuffle(perguntas)

    pergunta_validacao = perguntas[0]

    for item in itens_categoria:

        pergunta = item["messages"][1]["content"]

        if pergunta == pergunta_validacao:
            validation_data.append(item)
        else:
            train_data.append(item)

print("Treino:", len(train_data))
print("Validação:", len(validation_data))

Treino: 40
Validação: 20


In [3]:
perguntas_train = {
    item["messages"][1]["content"]
    for item in train_data
}

perguntas_validation = {
    item["messages"][1]["content"]
    for item in validation_data
}

intersecao = perguntas_train.intersection(perguntas_validation)

print("Perguntas repetidas entre treino e validação:", len(intersecao))

Perguntas repetidas entre treino e validação: 0


In [4]:
arquivo_train = DATA_PROCESSED / "train.jsonl"
arquivo_validation = DATA_PROCESSED / "validation.jsonl"

with open(arquivo_train, "w", encoding="utf-8") as f:
    for item in train_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open(arquivo_validation, "w", encoding="utf-8") as f:
    for item in validation_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Arquivos criados:")
print("Treino:", arquivo_train)
print("Validação:", arquivo_validation)

Arquivos criados:
Treino: C:\Users\Diogo\tech-challenge-FIAP-fase3-assistente-medico\data\processed\train.jsonl
Validação: C:\Users\Diogo\tech-challenge-FIAP-fase3-assistente-medico\data\processed\validation.jsonl
